In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

In [ ]:
FILES = {
 "agent.yaml": "name: swe_coder\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Autonomous software engineer that fixes repository issues with minimal, verified patches.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\nskills:\n  - skills/code-tools\n  - skills/write-failing-test\n  - skills/exact-output\n  - skills/add-feature\n  - skills/errors-and-validation\ntools:\n  - get_status\n  - submit_patch\n",
 "configs/sampling.yaml": "temperature: 1.0\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: false\n",
 "prompts/system.md": "You are an autonomous senior Python engineer working inside a sandboxed checkout of a real open-source repository at /workspace.\nGoal: resolve the issue in the user message with the smallest correct patch, then call `submit_patch`.\n\n## Hard rules\n- Never edit, add or delete tests, `conftest.py`, `pytest.ini`, CI or packaging files. Hidden tests are applied after you finish.\n- Keep public APIs backward compatible unless the issue explicitly asks for a change.\n- Scratch files go to /tmp only. Anything left in /workspace becomes part of your patch.\n- Never add print or debug statements to source files: if time runs out, the working tree is graded as it is. Print from your /tmp test instead.\n- The environment is pre-built: do not try to install packages.\n- Always finish by calling `submit_patch`. A careful best-effort fix beats no patch.\n\n## Tools\nYou have no shell. Everything goes through the `code-tools` skill: call the tool `run_skill_script` with skill_name \"code-tools\", file_path \"scripts/ct.py\" and args (a list of strings). Call `load_skill` for it once at the start to see its commands (`outline`, `def`, `refs`, `grep`, `show`, `edit`, `write`, `run`). `run` executes only python, pytest and `git status`/`git diff`, e.g. args [\"run\", \"python -m pytest tests/test_x.py -q\"]. You have 32k tokens for the whole task: keep every output small.\n\n## Workflow: test first\n1. **Understand**: state the expected vs. actual behaviour in one or two sentences. Note the exact names, messages and values the issue uses. If one of these skills fits the issue, `load_skill` it: `exact-output`, `add-feature`, `errors-and-validation`.\n2. **Read the tests**: `grep` an identifier from the issue in `tests`, `outline` that test file and `show` the one or two tests closest to the issue. They show the public entry point, how the code is called and the style of messages. Hidden tests are usually added to this file.\n3. **Write a failing test**: first `load_skill` `write-failing-test`. Then `write` /tmp/test_issue.py in the same style, asserting what the issue asks for, with names and messages taken from the issue's own words. `run` it: `python -m pytest /tmp/test_issue.py -q`. It must fail, and for the reason the issue describes. If it passes or fails for another reason, fix the test before touching the code.\n4. **Find why it fails**: read the traceback, `show` the function it points to, and follow the call chain with `def` / `refs` / `show` until you reach the line where behaviour diverges from what the issue expects.\n5. **Fix**: choose the change with the minimal-fix ladder below and apply it with `edit`. Copy OLD exactly from `show` output, written as plain code (`\"\"\"` stays `\"\"\"`, never `\\\"\\\"\\\"`). Keep OLD short but unique. Hidden tests often match error messages, exception types and names exactly (`pytest.raises(..., match=...)`): build new messages and names from the issue's own words and title, not a paraphrase (issue says \"negative timeouts\" \u2192 `\"timeout must not be negative\"`, not `\"invalid timeout value\"`). If an edit fails, copy OLD from the closest text it prints; never resend the same OLD.\n6. **Re-run**: `run` /tmp/test_issue.py, which must pass now. Then `run` the test file from step 2 (`python -m pytest <tests/path> -x -q`), which must still pass. \"no tests ran\" (exit code 5) is not a pass. If an existing test fails only because it asserts the old behaviour the issue asks to change, that is expected: leave it, the hidden tests replace it. Any other failure: go back to step 4 with the new traceback.\n7. **Submit**: `run` `git diff`, make sure only intended source changes remain, then call `submit_patch`.\n\n## Minimal-fix ladder\nLazy about the fix, never about reading: trace the code the issue touches first, then stop at the first rung that works.\n1. Already supported (an option, flag, parameter)? Fix the wiring, don't add code.\n2. The codebase already has a helper, validator or pattern for this? Reuse it and copy its style.\n3. One guard in the shared function fixes every caller? Put it there (check its callers with `refs`), not in each caller.\n4. Only then: the smallest new code that works.\nNever cut what the issue asks for: every edge case it names, its error handling, backward compatibility.\nNo refactors, renames, new abstractions, reformatting or comment churn. A 3-line diff beats a 30-line one.\n\n## Budget\n- Most fixes need 10-20 tool calls. Once both test runs in step 6 pass, submit; don't polish.\n- Re-running a command without editing code in between returns the same output, so never do it. After an unexpected result, `show` the code you just changed and fix it, or change approach.\n- Call `get_status` every ~8 tool calls. When less than 25% of time remains, go straight to Fix \u2192 Re-run \u2192 Submit.\n- Keep outputs short: `pytest -q`, and `show` one function or a few lines at a time.\n",
 "skills/add-feature/SKILL.md": "---\nname: add-feature\ndescription: For issues asking for new behaviour (a new parameter, option, function or deprecation), and for issues written as pull-request descriptions rather than bug reports.\n---\n- The title and description are the spec; skip template text, checklists and links. Every name, default and message they mention is what hidden tests will use.\n- New parameter: give it a default that keeps the old behaviour, then thread it through every layer from the public entry point down to where it acts (check the chain with `refs`). Missing one layer is the most common failure.\n- Add new keyword arguments at the end; never reorder existing parameters.\n- Mirror a sibling: if a similar option already exists, copy its whole path (signature, docstring line, validation, the place it is used).\n- Export new public names where their siblings are exported (`__init__.py`, `__all__`).\n- Deprecation: keep the old behaviour working and warn with `warnings.warn(\"...\", DeprecationWarning, stacklevel=2)`, worded as in the issue.\n- Only change source code: no docs, changelogs or new test files.\n",
 "skills/code-tools/SKILL.md": "---\nname: code-tools\ndescription: Low-context Python navigation (file outline, go-to-definition including async defs, references, exact symbol source) plus reliable edit/write commands that tolerate quote and backslash escaping and refuse edits that break syntax. Use it for all reading and editing.\n---\nCall the tool `run_skill_script` with skill_name \"code-tools\", file_path \"scripts/ct.py\" and args (a list of strings).\nPaths are relative to /workspace. Output is compact; prefer it over reading whole files.\n\n| args | result |\n|---|---|\n| `[\"outline\", \"pkg/mod.py\"]` | classes/functions with line ranges. Use before reading a file |\n| `[\"def\", \"Name\"]` or `[\"def\", \"Class.method\"]` | where it is defined (async defs included) |\n| `[\"refs\", \"name\"]` | every line using it: callers, imports, attributes |\n| `[\"show\", \"pkg/mod.py\", \"Class.method\"]` or `[\"show\", \"pkg/mod.py\", \"120-160\"]` | exact source, no line numbers. Copy OLD text from here |\n| `[\"edit\", \"pkg/mod.py\", OLD, NEW]` | replace the exact text OLD with NEW (5th arg `\"all\"` replaces every occurrence). Not saved if the result is invalid Python. On a miss it prints the closest text: copy OLD from that |\n| `[\"grep\", \"PATTERN\"]` or `[\"grep\", \"PATTERN\", \"tests\"]` | regex search in .py files, at most 40 `file:line: text` hits |\n| `[\"run\", \"python -m pytest tests/test_x.py -q\"]` | run python, pytest or `git status`/`git diff` in /workspace; shows exit code and the end of the output. No search commands |\n| `[\"write\", \"/tmp/repro.py\", CONTENT]` | create or overwrite a file. Scratch files go in /tmp: anything in /workspace is part of your patch |\n\nWrite OLD/NEW as plain code, exactly as `show` printed it: `\"\"\"` stays `\"\"\"`, never `\\\"\\\"\\\"`.\n",
 "skills/code-tools/scripts/ct.py": "\"\"\"Compact code navigation + escape-tolerant editing for /workspace (stdlib only).\n\nct.py outline FILE              classes/functions with line ranges\nct.py def NAME                  where NAME (or Class.method) is defined, incl. async\nct.py refs NAME                 lines that use NAME\nct.py show FILE SYMBOL|A-B      exact source of a symbol or line range\nct.py edit FILE OLD NEW [all]   replace OLD with NEW (tolerates \\\\\" style escaping), syntax-checked\nct.py write PATH CONTENT        create/overwrite a file (/tmp allowed)\nct.py run \"COMMAND\"             run python / pytest / git status|diff in /workspace (no searching)\nct.py grep PATTERN [DIR]        regex search in .py files, max 40 matching lines\n\"\"\"\n\nimport ast\nimport codecs\nimport difflib\nimport os\nimport re\nimport shlex\nimport subprocess\nimport sys\n\nROOT = \"/workspace\"\nSKIP = {\".git\", \"__pycache__\", \"node_modules\", \"build\", \"dist\", \".venv\", \"venv\", \".tox\"}\nCAP = 4000  # ~1k tokens per call: the scorer has a 32k context for the whole task\n\n\ndef out(text):\n    print(text if len(text) <= CAP else text[:CAP] + f\"\\n... [cut at {CAP} chars; narrow the query]\")\n\n\ndef path_of(p):\n    p = p.strip()\n    return p if p.startswith(\"/tmp\") else os.path.join(ROOT, p.removeprefix(\"/workspace/\").lstrip(\"/\"))\n\n\ndef rel(p):\n    return os.path.relpath(p, ROOT)\n\n\ndef py_files(containing=None):\n    for d, dirs, files in os.walk(ROOT):\n        dirs[:] = sorted(x for x in dirs if x not in SKIP and not x.startswith(\".\"))\n        for f in sorted(files):\n            if f.endswith(\".py\") and not f.startswith(\".\"):  # skips ADK's .adk_exec_*.py (holds this skill + args)\n                p = os.path.join(d, f)\n                if containing is None or containing in open(p, errors=\"ignore\").read():\n                    yield p\n\n\ndef symbols(tree):\n    \"\"\"(qualname, node) for every class/function, nested ones included.\"\"\"\n    stack = [(\"\", n) for n in tree.body]\n    while stack:\n        prefix, n = stack.pop(0)\n        if isinstance(n, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):\n            q = prefix + n.name\n            yield q, n\n            stack += [(q + \".\", c) for c in n.body]\n\n\ndef start(n):\n    return min([d.lineno for d in getattr(n, \"decorator_list\", [])] + [n.lineno])\n\n\ndef head(n):\n    if isinstance(n, ast.ClassDef):\n        return f\"class {n.name}({', '.join(ast.unparse(b) for b in n.bases)})\"\n    kw = \"async def\" if isinstance(n, ast.AsyncFunctionDef) else \"def\"\n    sig = ast.unparse(n.args)\n    return f\"{kw} {n.name}({sig if len(sig) < 90 else sig[:87] + '...'})\"\n\n\ndef parse(p):\n    try:\n        return ast.parse(open(p, errors=\"ignore\").read())\n    except SyntaxError:\n        return None\n\n\ndef matches(q, name):\n    return q == name or q.endswith(\".\" + name)\n\n\ndef outline(f):\n    tree = parse(path_of(f))\n    if tree is None:\n        return out(f\"{f}: cannot parse\")\n    lines = [f\"{'  ' * q.count('.')}L{start(n)}-{n.end_lineno} {head(n)}\" for q, n in sorted(symbols(tree), key=lambda s: start(s[1]))]\n    out(\"\\n\".join(lines) or \"no classes or functions\")\n\n\ndef find_def(name):\n    hits = []\n    for p in py_files(name.split(\".\")[-1]):\n        tree = parse(p)\n        for q, n in symbols(tree) if tree else []:\n            if matches(q, name):\n                hits.append(f\"{rel(p)}:{n.lineno}-{n.end_lineno}  {q}  {head(n)}\")\n        for n in tree.body if tree else []:  # module-level constants / aliases\n            if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in n.targets):\n                hits.append(f\"{rel(p)}:{n.lineno}  {ast.unparse(n)[:120]}\")\n    out(\"\\n\".join(hits[:40]) or f\"no definition of {name}\")\n\n\ndef refs(name):\n    short, hits = name.split(\".\")[-1], []\n    for p in py_files(short):\n        tree = parse(p)\n        src = open(p, errors=\"ignore\").read().splitlines()\n        seen = set()\n        for n in ast.walk(tree) if tree else []:\n            hit = (isinstance(n, ast.Name) and n.id == short) or (isinstance(n, ast.Attribute) and n.attr == short) \\\n                or (isinstance(n, ast.alias) and n.name.split(\".\")[-1] == short)\n            if hit and n.lineno not in seen:\n                seen.add(n.lineno)\n                hits.append(f\"{rel(p)}:{n.lineno}: {src[n.lineno - 1].strip()[:110]}\")\n    extra = f\"\\n... {len(hits) - 50} more\" if len(hits) > 50 else \"\"\n    out(\"\\n\".join(sorted(hits)[:50]) + extra if hits else f\"no references to {short}\")\n\n\ndef show(f, what):\n    p = path_of(f)\n    lines = open(p, errors=\"ignore\").read().splitlines()\n    if what.replace(\"-\", \"\").replace(\":\", \"\").isdigit():\n        a, _, b = what.replace(\":\", \"-\").partition(\"-\")\n        a, b = int(a), int(b or a)\n    else:\n        tree = parse(p)\n        found = [(start(n), n.end_lineno) for q, n in symbols(tree) if matches(q, what)] if tree else []\n        if not found:\n            return out(f\"{what} not found in {f}; try: ct.py outline {f}\")\n        a, b = found[0]\n    out(f\"# {rel(p)} lines {a}-{b} (copy text from here for edits; no line numbers inside)\\n\" + \"\\n\".join(lines[a - 1:b]))\n\n\n# The text as sent, then the usual JSON-escaping slips: \\\\\" for \", halved or doubled backslashes.\nTRANSFORMS = [\n    lambda s: s,\n    lambda s: s.replace('\\\\\"', '\"').replace(\"\\\\'\", \"'\"),\n    lambda s: s.replace(\"\\\\\\\\\", \"\\\\\"),\n    lambda s: s.replace(\"\\\\\", \"\\\\\\\\\"),\n    lambda s: codecs.escape_decode(s.encode())[0].decode(),\n]\n\n\ndef variants(s):\n    \"\"\"(transform, transformed text) pairs; the same transform is later applied to NEW.\"\"\"\n    for t in TRANSFORMS:\n        try:\n            yield t, t(s)\n        except Exception:\n            pass\n\n\ndef syntax_error(p, text):\n    if not p.endswith(\".py\"):\n        return None\n    try:\n        compile(text, p, \"exec\")\n    except SyntaxError as e:\n        return f\"line {e.lineno}: {e.msg}\"\n\n\ndef is_test_file(p):\n    \"\"\"Test files are reset before grading; an edit to one makes the hidden tests fail to apply.\"\"\"\n    if p.startswith(\"/tmp\"):\n        return False\n    name, parts = os.path.basename(p), rel(p).split(os.sep)\n    return (name.startswith(\"test_\") or name.endswith(\"_test.py\") or name in {\"conftest.py\", \"pytest.ini\"}\n            or \"tests\" in parts[:-1] or \"test\" in parts[:-1])\n\n\nTEST_REFUSED = (\"ERROR: refused to change a test file. Test files are reset before grading, and a changed one makes the \"\n                \"hidden tests fail to apply, so the whole task fails. If an existing test asserts the old behaviour the \"\n                \"issue asks to change, leave it: the hidden tests replace it. Change source code only.\")\n\n\ndef edit(f, old, new, allow_all=False):\n    p = path_of(f)\n    if is_test_file(p):\n        return out(TEST_REFUSED)\n    src = open(p, errors=\"ignore\").read()\n    for t, v in variants(old):\n        if v and v in src:\n            count = src.count(v)\n            if count > 1 and not allow_all:\n                at = [src[:i].count(\"\\n\") + 1 for i in _positions(src, v)]\n                return out(f\"ERROR: OLD occurs {count} times (lines {at}); add surrounding lines to make it unique, or pass 'all'\")\n            new_v = t(new)\n            result = src.replace(v, new_v)\n            err = syntax_error(p, result)\n            if err:\n                return out(f\"ERROR: not saved, the edit would make {rel(p)} invalid Python ({err}). Check quotes/indentation in NEW.\")\n            open(p, \"w\").write(result)\n            diff = difflib.unified_diff(src.splitlines(), result.splitlines(), rel(p), rel(p), lineterm=\"\", n=1)\n            return out(f\"OK: {count} replacement(s) in {rel(p)}\" + (\"\" if v == old else \" (escaping in OLD/NEW was normalized)\")\n                       + \"\\n\" + \"\\n\".join(list(diff)[2:]))\n    lines, n = src.splitlines(), max(1, old.count(\"\\n\") + 1)\n    best = max(range(max(1, len(lines) - n + 1)), key=lambda i: difflib.SequenceMatcher(None, \"\\n\".join(lines[i:i + n]), old).ratio())\n    out(f\"ERROR: OLD not found in {rel(p)}. Closest text is lines {best + 1}-{best + n}; copy OLD exactly from it:\\n\"\n        + \"\\n\".join(lines[best:best + n]))\n\n\ndef _positions(s, sub):\n    i = s.find(sub)\n    while i >= 0:\n        yield i\n        i = s.find(sub, i + 1)\n\n\ndef write(f, content):\n    p = path_of(f)\n    if is_test_file(p):\n        return out(TEST_REFUSED)\n    for _, v in variants(content):\n        if not syntax_error(p, v):\n            content = v\n            break\n    os.makedirs(os.path.dirname(p), exist_ok=True)\n    open(p, \"w\").write(content)\n    note = \"\" if p.startswith(\"/tmp\") else \" (inside /workspace: this file WILL be in your patch)\"\n    out(f\"OK: wrote {p} ({len(content)} chars){note}\" + (f\"\\nWARNING: {syntax_error(p, content)}\" if syntax_error(p, content) else \"\"))\n\n\ndef grep(pattern, where=\".\"):\n    try:\n        rx = re.compile(pattern)\n    except re.error:\n        rx = re.compile(re.escape(pattern))\n    base = os.path.normpath(os.path.join(ROOT, where.removeprefix(\"/workspace/\").lstrip(\"/\")))\n    hits = []\n    for p in py_files():\n        if p.startswith(base):\n            for i, line in enumerate(open(p, errors=\"ignore\"), 1):\n                if rx.search(line):\n                    hits.append(f\"{rel(p)}:{i}: {line.strip()[:120]}\")\n    more = f\"\\n... {len(hits) - 40} more; narrow PATTERN or DIR\" if len(hits) > 40 else \"\"\n    out(\"\\n\".join(hits[:40]) + more if hits else f\"no match for {pattern!r} in {where}\")\n\n\nRUNNABLE = {\"python\", \"python3\", \"pytest\"}\nGIT_OK = {\"status\", \"diff\"}\n\n\ndef run(cmd):\n    \"\"\"Run/verify only. Searching belongs to code_analyzer, whose context is discarded afterwards.\"\"\"\n    try:\n        words = shlex.split(cmd)\n    except ValueError as e:\n        return out(f\"ERROR: {e}\")\n    first = os.path.basename(words[0]) if words else \"\"\n    ok = first in RUNNABLE or (first == \"git\" and len(words) > 1 and words[1] in GIT_OK)\n    if not ok or any(w in {\"grep\", \"egrep\", \"rg\", \"find\", \"cat\", \"ls\", \"head\", \"tail\", \"sed\", \"awk\"} for w in words):\n        return out(\"ERROR: `run` only runs python, pytest and git status/diff. To search or read code, ask \"\n                   \"code_analyzer, or use outline/def/refs/show.\")\n    r = subprocess.run(cmd, shell=True, cwd=ROOT, capture_output=True, text=True, timeout=240)\n    text = (r.stdout + (\"\\n[stderr]\\n\" + r.stderr if r.stderr.strip() else \"\")).strip()\n    if len(text) > CAP:  # keep the end: pytest summaries and tracebacks are at the bottom\n        text = f\"... [first {len(text) - CAP} chars cut]\\n\" + text[-CAP:]\n    print(f\"exit code {r.returncode}\\n{text}\")\n\n\ndef main(argv):\n    cmds = {\"outline\": (outline, 1), \"def\": (find_def, 1), \"refs\": (refs, 1), \"show\": (show, 2),\n            \"edit\": (edit, 3), \"write\": (write, 2), \"run\": (run, 1), \"grep\": (grep, 1)}\n    while argv and argv[0] in {\"run_skill_script\", \"code-tools\", \"scripts/ct.py\", \"ct.py\"}:\n        argv = argv[1:]  # the tool's own parameters sometimes get pasted into args\n    if not argv or argv[0] not in cmds or len(argv) - 1 < cmds[argv[0]][1]:\n        return out(__doc__)\n    fn, n = cmds[argv[0]]\n    args = argv[1:1 + n]\n    if argv[0] == \"edit\" and len(argv) > 4:\n        args.append(argv[4] == \"all\")\n    if argv[0] == \"grep\" and len(argv) > 2:\n        args.append(argv[2])\n    try:\n        fn(*args)\n    except (OSError, ValueError, subprocess.TimeoutExpired) as e:\n        out(f\"ERROR: {e}\")\n\n\nif __name__ == \"__main__\":\n    main(sys.argv[1:])\n",
 "skills/errors-and-validation/SKILL.md": "---\nname: errors-and-validation\ndescription: For issues about input that should raise (or stop raising) an error, about error messages or exception types, or about validation rules.\n---\n- Hidden tests usually use `pytest.raises(<Type>, match=<message>)`: both the exception type and the message words matter.\n- Exception type: follow the issue; otherwise use the type that sibling checks in the same module raise (`grep \"raise \" <file>`).\n- Message: state the rule in the issue's own words (title and description), in the style of the module's existing messages (quoting, field names).\n- Validate at the one place every input passes through (constructor, validator, setter, parser), not in each caller.\n- Cover every invalid form the issue names, and keep valid inputs (None, defaults) working.\n- If something should stop raising, narrow the condition of the check that raises; don't wrap callers in try/except.\n",
 "skills/exact-output/SKILL.md": "---\nname: exact-output\ndescription: For issues about printed, rendered, serialized or formatted output (text, tables, markup, JSON, headers, repr, docs examples) - hidden tests compare exact strings.\n---\n- Tests compare the whole string: spaces, trailing newlines, ordering, quoting and escape/ANSI codes all count.\n- Reproduce by capturing the real output and printing `repr()` of it; compare it character by character with what the issue says it should be.\n- Find where the text is produced (`grep` a fixed fragment of the output), not where it is printed.\n- Render with the same options the closest existing test uses (width, color settings, flags); output often depends on them.\n- Change only the part the issue names. Don't reformat anything else: neighbouring tests compare other exact strings.\n- After the fix, run the whole test file; formatting fixes often break a neighbouring expectation.\n",
 "skills/write-failing-test/SKILL.md": "---\nname: write-failing-test\ndescription: How to write /tmp/test_issue.py so it fails for the issue's reason - exact exception messages, captured output, warnings, async code, and working without the repo's fixtures.\n---\n- Start from a copy, never from scratch: `show` the top of the closest test file (imports and module-level setup) and the closest test function, and copy them into /tmp/test_issue.py. Then change only the input and the assertion.\n- Framework code (web apps, CLIs, ORMs, plugins) is tested through the setup the repo's tests already use (the app/route/test client, runner or session they build). Copy that setup exactly; an invented setup usually tests something else.\n- Fixtures from the repo's `conftest.py` are NOT available in /tmp: copy the fixture's body into the test instead.\n- Exceptions: `with pytest.raises(ValueError, match=re.escape(\"exact message\")):` (match is a regex searched in `str(exc)`, so escape it).\n- Output: capture it into a string and compare the whole thing (`assert out == \"...\"`). When it fails, print `repr(out)` once to see spaces, newlines and escape codes.\n- Warnings: `with pytest.warns(DeprecationWarning, match=\"...\"):`.\n- Async code: call `asyncio.run(main())` inside a normal test function; no plugin needed.\n- Several inputs: `@pytest.mark.parametrize(\"value\", [...])` with every case the issue names.\n- Before the fix it must fail with the problem the issue describes. An ImportError, NameError or TypeError raised by the test itself means the test is wrong: fix the test.\n- If it passes before the fix, that path already works: the bug is on another way into the same feature. Features usually have several (a single parameter vs a model/class/config object that groups them, sync vs async, one value vs a list, a function vs a method). List them, then test the next one. The closest existing test file's name often says which path the issue is about.\n- After two passing attempts, stop writing tests: `grep` where the feature's option is read in the source and check each code path that reads it.\n- Keep one test. Once /tmp/test_issue.py fails for the issue's reason, it is your only check: don't write other test files, don't mock internal functions, don't test third-party libraries.\n- Trust the environment: it is offline and correctly installed. If a library seems broken, your usage is wrong (for example a real network call).\n- A fix that doesn't change the test result is on the wrong line: `show` the rest of the function handling the value (especially loops after your edit) and trace where the bad value still gets through.\n"
}
from pathlib import Path
from swegemma.models import load_tasks
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'agent'
for rel, text in FILES.items():
    (AGENT_DIR / rel).parent.mkdir(parents=True, exist_ok=True)
    (AGENT_DIR / rel).write_text(text)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
print(len(FILES), 'bundle files;', len(tasks), 'tasks')


In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    default_chat_template_kwargs={'enable_thinking': True},
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

In [ ]:
import asyncio, concurrent.futures, json, time
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(fn, *a, **k):
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(fn(*a, **k))).result()

TASK_IDS = {'requests_7309', 'fastapi_15589'}
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
    timeout_seconds=300, max_time_minutes=4, max_tool_calls=24, max_turns=None,
    limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                                                    token_threshold=14336, event_retention_size=5),
    graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels', verbose=False,
)
evaluator = Evaluator(eval_config)
rows = []
for i, task in enumerate([t for t in tasks if t.instance_id in TASK_IDS], 1):
    t0 = time.time()
    r = run_sync(evaluator.evaluate_task, task=task, task_index=i, total_tasks=len(TASK_IDS))
    rows.append(dict(id=task.instance_id, resolved=r.resolved, exit=r.test_exit_code, patch=len(r.agent_patch or ''),
                     tool_calls=r.tool_calls, secs=round(time.time() - t0), error=(r.error or '')[:300]))
    print(json.dumps(rows[-1]), flush=True)
(WORKING_DIR / 'gpu_eval.json').write_text(json.dumps(rows, indent=1))


In [ ]:
import re
for log in sorted((WORKING_DIR / 'results' / 'logs').glob('*.log')):
    text = re.sub(r'\x1b\[[0-9;]*m', '', log.read_text(errors='ignore'))
    print('#' * 20, log.name, f'({len(text)} chars)')
    print(text[-6000:])
